## Загрузка датасета GorillaHard2 на HuggingFace

Ноутбук собирает `test.json` и `shots.json` из `datasets/GorillaHard2/` в `datasets.DatasetDict`
и заливает его на 🤗 Hub по указанному пути.

Что здесь специфично для этого датасета:
* поле `instruction` в локальных файлах — это **индекс** промпта в `dataset_meta.json["prompts"]`;
  перед заливкой он заменяется на сам текст промпта (общее правило MERA);
* `inputs.context` и `inputs.tools` — сырой файл и JSON-каталог, оба полны фигурных скобок,
  поэтому промпт собирается заменой подстрок, а не `str.format`;
* у каждого вопроса ровно одна строка: 1169 вопросов — 1169 строк теста, плюс 5 в `shots`;
* пятьдесят пять строк — ходы двадцати пяти диалогов. Историю хода собирает сэмплер задачи по
  `meta.dialog_id` и `meta.turn_id`, а текстом ответа в истории служит `outputs` предыдущего хода;
* `outputs` — это и есть правильный ответ (один из пяти конвертов), по нему считаются
  содержательные метрики. Про режим приватной заливки читайте предупреждение ниже;
* перед заливкой прогоняется проверка целостности: каждый эталонный ответ должен получать
  `sample_pass_rate = 1.0` у того же скорера, которым считаются метрики.


In [1]:
import json
import os
import sys

import datasets
from tqdm import tqdm

c:\Users\arorlov\.conda\envs\default\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\arorlov\.conda\envs\default\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


### Подготовка данных

#### WARNING!

Если ваш датасет является __ПРИВАТНЫМ__, оставьте `MY_DATASET_IS_PRIVATE_LETS_HIDE_ANSWERS` равным `True`.
Иначе поставьте `False`. Этот флаг дальше используется, чтобы стереть ответы перед загрузкой на ХФ.
На ХФ даже приватно не должно лежать датасетов с ответами!

⚠️ Здесь `outputs` — это сам правильный ответ, а не свидетельство решаемости: по нему сверяются вид
конверта, инструмент и аргументы. Если стереть ответы, то по копии с Hub можно посчитать только
`format_pass_rate` и `constraint_pass_rate` — остальные метрики скорер осознанно не выдаёт
(см. `benchmark_tasks/gorillahard2/utils.py`), чтобы отсутствие эталона не выглядело как провал
модели.

⚠️ Второе следствие, про многоходовость: историю диалога сэмплер собирает из `outputs` предыдущих
ходов. На копии со стёртыми ответами история приходит с пустыми репликами ассистента, то есть
`dialog_pass_rate` на ней не измеряется, а ходы после первого становятся недоопределёнными.
Полную оценку в обоих случаях проводит организатор бенчмарка на непубличной копии с ответами.


In [2]:
MY_DATASET_IS_PRIVATE_LETS_HIDE_ANSWERS = True

Параметр `path_to_data` — путь ДО файлов `shots.json` и `test.json`.

Параметр `path_to_meta` — путь ДО `dataset_meta.json`.

Пути ниже указаны относительно расположения ноутбука в `utils/`; поменяйте, если запускаете из другого места.

In [3]:
path_to_data = "."
path_to_meta = "."
path_to_task = "../benchmark_tasks/gorillahard"

Сплиты и мета лежат в формате JSON.

In [4]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data

#### Подгрузка данных

In [5]:
shots = load_json(os.path.join(path_to_data, "shots.json"))["data"]
test = load_json(os.path.join(path_to_data, "test.json"))["data"]
meta = load_json(os.path.join(path_to_meta, "dataset_meta.json"))

print(f"shots: {len(shots)}, test: {len(test)}")

shots: 5, test: 1169


Из меты для датасета нужны только промпты.

In [6]:
prompts = meta["prompts"]
len(prompts)

5

#### Обработка полей датасета

На ХФ загружается датасет, где у КАЖДОГО сэмпла вместо числа в поле `instruction` стоит промпт.
Число указывает, какой по индексу взять промпт из секции с промптами в мете датасета.

Ячейка идемпотентна: если её случайно выполнить дважды, строки не будут перезаписаны повторно.

In [7]:
def resolve_prompts(split):
    for card in split:
        if isinstance(card["instruction"], int):
            card["instruction"] = prompts[card["instruction"]]


resolve_prompts(shots)
resolve_prompts(test)

print(test[0]["instruction"][:200], "...")

Требуется обработать запрос и вернуть ответ строго в формате из блока «Формат ответа». Когда вопрос закрывается одним вызовом, из каталога выбирается ровно один инструмент, и его аргументы заполняются ...


#### Проверка целостности перед заливкой

Собираем промпт ровно так, как это будет делать lm-eval (`utils.doc_to_text` — замена подстрок,
не `str.format`), и прогоняем эталонный ответ через тот же скорер, которым считается метрика.
Если что-то не сходится — на Hub такой датасет заливать нельзя.

In [8]:
sys.path.insert(0, os.path.abspath(path_to_task))
import utils as U

bad_prompt, bad_gold = [], []
balance_items, dialog_items = [], []
for card in tqdm(shots + test):
    rendered = U.doc_to_text(card)
    if not rendered.strip() or "{question}" in rendered or "{tools}" in rendered:
        bad_prompt.append(card["meta"]["id"])
    res = U.process_results(card, [card["outputs"]])
    if res.get("sample_pass_rate") != 1.0:
        failed = [c["name"] for c in U.score_response(card, card["outputs"])["format_checks"]
                  if not c["pass"]]
        bad_gold.append((card["meta"]["id"], failed))
    balance_items.append(res["balance_score"])
    if "dialog_pass_rate" in res:
        dialog_items.append(res["dialog_pass_rate"])

print("промптов, которые не собираются:", len(bad_prompt))
print("эталонных ответов, не проходящих проверку:", len(bad_gold))
assert not bad_prompt and not bad_gold, (bad_prompt[:5], bad_gold[:5])

# Обе главные метрики на эталоне обязаны давать единицу, каждая по своей
# агрегации: balance_score — по рычагам сложности, dialog_pass_rate — по
# диалогам. Единица у первой метрики их не гарантирует: у них другие группы.
assert U.balance_aggregation(balance_items) == 1.0
assert U.dialog_aggregation(dialog_items) == 1.0
print("balance_score и dialog_pass_rate на эталоне: 1.0")


  0%|          | 0/1174 [00:00<?, ?it/s]

100%|██████████| 1174/1174 [00:00<00:00, 12033.19it/s]

промптов, которые не собираются: 0
эталонных ответов, не проходящих проверку: 0
balance_score и dialog_pass_rate на эталоне: 1.0


#### Убираем ответы для приватных задач

Надеемся, вы поставили в начале ноутбука корректное значение `MY_DATASET_IS_PRIVATE_LETS_HIDE_ANSWERS`.

Если там стоит `True`, то в `test` сплите ответы на все задания стираются. Вместо них остаётся пустая
строка, чтобы вы случайно не пушнули на ХФ датасет с заполненными ответами, и они не утекли.
Ответы в `shots` сохраняются: это few-shot примеры, они и должны быть видны модели.

In [9]:
def hide_answers(dataset_split: list):
    for card in tqdm(dataset_split):
        card["outputs"] = ""

In [10]:
if MY_DATASET_IS_PRIVATE_LETS_HIDE_ANSWERS:
    hide_answers(test)

100%|██████████| 1169/1169 [00:00<?, ?it/s]


### Создаем датасет для загрузки на ХФ

#### Аннотация полей датасета

В `features` повторяется структура КАЖДОГО сэмпла датасета с описанием формата данных в каждом поле.

Четыре места, на которые стоит посмотреть:
* `inputs.tools` — **строка** с JSON-каталогом, а не вложенная структура: у инструментов разное
  число параметров, фиксированной схемой их не описать, и модель всё равно видит каталог текстом;
* `inputs.context` — строка, у части вопросов пустая: они опираются на карточку задачи, а не на
  приложенный файл. Поле должно присутствовать всегда, иначе схема разъедется;
* поля `format_trap`, `cost_pick` и `injected_tool` в `meta.categories` — аналитические. По двум
  последним считаются `cost_optimal_rate` и `injection_resistance_rate`; модели, как и вся `meta`,
  они не показываются. Поля `annotation` в схеме нет: человеческая разметка не проводилась;
* поля `dialog_id`, `turn_id`, `n_turns`, `needs_history` и `wording` в `meta` — структура
  многоходовости. По ним сэмплер задачи собирает историю хода, а по `n_turns` считается
  `dialog_pass_rate`. У однокадрового вопроса `dialog_id` совпадает с `base_id`, `turn_id` равен
  нулю, `n_turns` единице, `needs_history` пуст — второго вида строк в датасете нет.


In [11]:
features = datasets.Features({
    "instruction": datasets.Value("string"),
    "inputs": {
        "question": datasets.Value("string"),
        "context": datasets.Value("string"),
        "tools": datasets.Value("string"),
        "format": datasets.Value("string"),
    },
    "outputs": datasets.Value("string"),
    "meta": {
        "id": datasets.Value("int32"),
        "base_id": datasets.Value("string"),
        "dialog_id": datasets.Value("string"),
        "turn_id": datasets.Value("int32"),
        "n_turns": datasets.Value("int32"),
        "needs_history": datasets.Value("string"),
        "wording": datasets.Value("int32"),
        "categories": {
            "language": datasets.Value("string"),
            "difficulty": datasets.Value("string"),
            "family": datasets.Value("string"),
            "lever": datasets.Value("string"),
            "answer_kind": datasets.Value("string"),
            "n_tools": datasets.Value("int32"),
            "n_files": datasets.Value("int32"),
            "has_context": datasets.Value("string"),
            "corpus_format": datasets.Value("string"),
            "format_trap": datasets.Value("string"),
            "cost_pick": datasets.Value("string"),
            "injected_tool": datasets.Value("string"),
        },
    },
})

#### Создание датасетов для каждого сплита

Сэмплы тяжёлые — в `inputs.context` лежит целый файл, медиана промпта около 18 тысяч символов, —
поэтому тестовый сплит собирается кусками по `STEP` сэмплов: так конвертация идёт заметно быстрее,
чем одним вызовом на все 1169.


In [12]:
shots_ds = datasets.Dataset.from_list(shots, features=features)

STEP = 50
chunks = []
for i in tqdm(range(0, len(test), STEP)):
    chunks.append(datasets.Dataset.from_list(test[i: i + STEP], features=features))
test_ds = datasets.concatenate_datasets(chunks)

shots_ds, test_ds

100%|██████████| 24/24 [00:00<00:00, 124.81it/s]


(Dataset({
     features: ['instruction', 'inputs', 'outputs', 'meta'],
     num_rows: 5
 }),
 Dataset({
     features: ['instruction', 'inputs', 'outputs', 'meta'],
     num_rows: 1169
 }))

##### Проверка

Проверим, что сборка прошла успешно — ничего не потеряно, не продублировано и не переехало.

In [13]:
# количество вопросов до конвертации и после совпадает
assert len(test) == len(test_ds) and len(shots) == len(shots_ds)

# id вопросов сходятся и остаются сквозными: 1..5 в shots, 6..1174 в test
assert [c["meta"]["id"] for c in test] == [c["meta"]["id"] for c in test_ds]
assert [c["meta"]["id"] for c in shots] == [c["meta"]["id"] for c in shots_ds]
ids = sorted(c["meta"]["id"] for c in shots + test)
assert ids == list(range(1, len(ids) + 1))

# каталог инструментов пережил конвертацию и всё ещё парсится
assert all(json.loads(c["inputs"]["tools"]) for c in test_ds)

# конверты ответа пережили конвертацию: их пять, и ни один не должен
# потеряться при сборке DatasetDict
kinds = {c["meta"]["categories"]["answer_kind"] for c in test_ds}
assert kinds == {"tool_call", "plan", "calls", "clarify", "abstain"}, kinds

# у каждого вопроса ровно одна строка, и формулировки разложены по датасету:
# дубль строки или потерянная формулировка означали бы, что состав разъехался
base_ids = [c["meta"]["base_id"] for c in test_ds]
assert len(set(base_ids)) == len(base_ids)
wordings = sorted({c["meta"]["wording"] for c in test_ds})
assert wordings == list(range(len(prompts))), wordings

# многоходовость: у каждого диалога ходы нумерованы подряд с нуля и лежат
# целиком в тестовом сплите — иначе сэмплеру будет не из чего собрать историю
turns = {}
for c in test_ds:
    turns.setdefault(c["meta"]["dialog_id"], []).append(c["meta"]["turn_id"])
for dialog_id, seen in turns.items():
    assert sorted(seen) == list(range(len(seen))), (dialog_id, sorted(seen))
multi = [d for d, seen in turns.items() if len(seen) > 1]
print("OK, конверты:", sorted(kinds), "| диалогов:", len(multi),
      "| строк:", len(test_ds))


OK, конверты: ['abstain', 'calls', 'clarify', 'plan', 'tool_call'] | диалогов: 25 | строк: 1169


#### Собираем сплиты в один датасет

In [14]:
dataset = datasets.DatasetDict({"shots": shots_ds, "test": test_ds})
dataset

DatasetDict({
    shots: Dataset({
        features: ['instruction', 'inputs', 'outputs', 'meta'],
        num_rows: 5
    })
    test: Dataset({
        features: ['instruction', 'inputs', 'outputs', 'meta'],
        num_rows: 1169
    })
})

### Загрузка датасета на ХФ

Для загрузки на ХФ понадобятся:
- Токен — строка с ключом, дающим право записи в репозиторий.
- Путь для записи — аккаунт и название датасета. Название пишите ровно так, как оно заявлено в мете
  (`dataset_meta.json["dataset_name"]`), регистр имеет значение.

Советуем сначала залить всё приватно и выслать на почту mera@a-ai.ru токен и путь для верификации.

In [ ]:
from dotenv import load_dotenv

# Загружаем переменные из .env файла
load_dotenv('../.env')

### TOKEN
token = os.getenv('HF_TOKEN')
if token is None:
    raise ValueError("HF_TOKEN not found in .env file")

### UPLOAD PATH — поменяйте на нужный вам путь
dataset_path_hub = "MERA-evaluation/GorillaHard"

# Чтобы предварительно посмотреть, как датасет будет выглядеть после заливки,
# можно сначала загрузить его в свой приватный репозиторий:
# dataset_path_hub = "<your-account>/GorillaHard"

### PRIVATE OR PUBLIC
upload_private = True

print("upload to:", dataset_path_hub, "| private:", upload_private)

upload to: MERA-evaluation/GorillaHard | private: True


In [16]:
dataset.push_to_hub(dataset_path_hub, private=upload_private, token=token)

Setting num_proc from 1 back to 1 for the shots split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 664.92ba/s]
Processing Files (1 / 1): 100%|██████████|  138kB /  138kB, 69.4kB/s  
New Data Upload: 100%|██████████|  138kB /  138kB, 69.4kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:03<00:00,  3.14s/ shards]
Setting num_proc from 1 back to 1 for the test split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 41.70ba/s]
Processing Files (1 / 1): 100%|██████████| 27.0MB / 27.0MB, 12.3MB/s  
New Data Upload: 100%|██████████| 27.0MB / 27.0MB, 12.3MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:03<00:00,  3.04s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/MERA-evaluation/GorillaHard/commit/380c9a44636944e84f34a622f523738f896ab301', commit_message='Upload dataset', commit_description='', oid='380c9a44636944e84f34a622f523738f896ab301', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/MERA-evaluation/GorillaHard', endpoint='https://huggingface.co', repo_type='dataset', repo_id='MERA-evaluation/GorillaHard'), pr_revision=None, pr_num=None)

### Проверка того, как датасет загрузился на ХФ

Загрузим датасет обратно и убедимся, что его увидит корректно любой, кто его скачает:
все поля на месте, содержание совпадает с исходным, а промпт по-прежнему собирается.

In [17]:
ds = datasets.load_dataset(dataset_path_hub, token=token)
ds

c:\Users\arorlov\.conda\envs\default\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\arorlov\.cache\huggingface\hub\datasets--MERA-evaluation--GorillaHard. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating test split: 100%|██████████| 1169/1169 [00:00<00:00, 39568.27 examples/s]


DatasetDict({
    shots: Dataset({
        features: ['instruction', 'inputs', 'outputs', 'meta'],
        num_rows: 5
    })
    test: Dataset({
        features: ['instruction', 'inputs', 'outputs', 'meta'],
        num_rows: 1169
    })
})

In [18]:
check = []
for idx, card in enumerate(ds["test"]):
    same_question = test[idx]["inputs"]["question"] == card["inputs"]["question"]
    same_tools = test[idx]["inputs"]["tools"] == card["inputs"]["tools"]
    check.append(same_question and same_tools)

all(check)

True

Финальная проверка: промпт собирается из скачанной с Hub копии — ровно то, что будет делать lm-eval.
После этого задачу можно запускать по HF-конфигу:

```bash
lm_eval --tasks gorillahard --include_path ./benchmark_tasks \
        --model hf --model_args pretrained=<model> --output_path ./out --log_samples
```

In [19]:
card = ds["test"][0]
print(U.doc_to_text(card)[:600], "...")

Требуется обработать запрос и вернуть ответ строго в формате из блока «Формат ответа». Когда вопрос закрывается одним вызовом, из каталога выбирается ровно один инструмент, и его аргументы заполняются по контексту. Величины, которые вопрос предлагает определить по приложенным файлам, — номер строки, количество, граница окна — вычисляются самостоятельно и подставляются в аргументы готовыми значениями. План из последовательных вызовов допускается только для данных, отсутствующих в контексте и получаемых первым вызовом; ссылка на результат шага записывается строкой "$1" ("$2" — на результат второ ...
